In [ ]:
#Phase 0: 需求澄清

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# 全局设置：先设 seaborn 风格，再覆盖中文字体（顺序不能反！）
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
#读取原始数据
df = pd.read_csv(r'D:\Ecommerce-Data-Analysis\data\Sample - Superstore.csv', encoding='cp1252')
df.head()

In [ ]:
#数据检查
# 规模与结构
print("\n【问题1】数据规模")
print(f"总数据行数：{df.shape[0]}")
print(f"总数据列数：{df.shape[1]}")
#原始数据类型
print("\n" + "="*60)
print("\n【问题2】各列数据类型")
dtypes_df = pd.DataFrame({
    '列名': df.columns,
    '数据类型': df.dtypes.astype(str)
})
print(dtypes_df.to_string(index=False))
print("\n" + "="*60)



In [ ]:
#缺失值
print("\n【问题3】各列缺失值统计")
missing_df = pd.DataFrame({
    '列名': df.columns,
    '缺失值数量': df.isnull().sum(),
    '缺失率(%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['缺失值数量'] > 0]
if len(missing_df) == 0:
    print("✅ 数据集中没有缺失值")
else:
    print(missing_df.to_string(index=False))
print("="*60)
print("🔍 重复数据检查报告")
print("="*60)

In [ ]:
# 重复行和重复订单号
print("\n【1/2】完全重复行检查")
# 检查所有列都完全相同的行
duplicate_rows = df[df.duplicated()]
duplicate_count = len(duplicate_rows)

if duplicate_count == 0:
    print("✅ 没有发现完全重复的行")
else:
    print(f"❌ 发现 {duplicate_count} 行完全重复的数据")
    print("\n重复行示例：")
    print(duplicate_rows.head().to_string())

    # 处理建议：删除完全重复行
    print("\n⚙️  自动删除完全重复行...")
    df = df.drop_duplicates(keep='first')
    print(f"✅ 删除完成，剩余数据行数：{len(df)}")

# ====================== 2. 重复订单号检查（重点） ======================
print("\n【2/2】重复订单号检查")
# 统计每个订单号出现的次数
order_id_counts = df['Order ID'].value_counts()
# 找出出现多次的订单号
duplicate_order_ids = order_id_counts[order_id_counts > 1]

print(f"总订单数量：{df['Order ID'].nunique()} 个")
print(f"包含多个商品的订单数量：{len(duplicate_order_ids)} 个")
print(f"单个订单最多包含商品数：{order_id_counts.max()} 个")

print("\n📌 重要说明：")
print("同一个订单号出现多次是**正常现象**")
print("因为一个订单通常会包含多个不同的商品，每行代表一个商品条目")
print("⚠️  绝对不要删除重复的订单号！这会丢失商品明细数据")

# 查看一个多商品订单的示例
if len(duplicate_order_ids) > 0:
    sample_order_id = duplicate_order_ids.index[0]
    print(f"\n📋 多商品订单示例（订单号：{sample_order_id}）：")
    sample_order = df[df['Order ID'] == sample_order_id]
    print(sample_order[['Order ID', 'Product Name', 'Sales', 'Quantity']].to_string(index=False))

print("\n" + "="*60)
print("✅ 重复数据检查完成")
print("="*60)
print("\n" + "="*60)

In [ ]:
#关键统计
print("\n【问题4】关键数值列统计分析")
key_columns = ['Sales', 'Profit', 'Discount']
stats_df = df[key_columns].describe().T[['min', '25%', '50%', '75%', 'max']]
stats_df.columns = ['最小值', '25分位数', '中位数', '75分位数', '最大值']
stats_df = stats_df.round(2)
print(stats_df.to_string())
print("\n⚠️  负值检查：")
for col in key_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"❌ 列 '{col}' 有 {negative_count} 个负值")
        if col == 'Profit':
            print(f"   （利润为负表示亏损订单，属于正常情况）")
    else:
        print(f"✅ 列 '{col}' 没有负值")
print("\n⚠️  离群值检查（3σ原则）：")
for col in key_columns:
    mean = df[col].mean()
    std = df[col].std()
    lower_bound = mean - 3 * std
    upper_bound = mean + 3 * std
    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_ratio = (outlier_count / len(df) * 100).round(2)
    print(f"列 '{col}'：{outlier_count} 个离群值，占比 {outlier_ratio}%")
    print(f"   正常范围：{lower_bound:.2f} ~ {upper_bound:.2f}")
    print("\n" + "="*60)

In [ ]:

# 保存转换前的数据类型用于对比
original_dtypes = df.dtypes.astype(str).to_dict()

# ====================== 1. 日期类型转换 ======================
print("\n【1/3】日期类型转换")
# 指定日期格式可以大幅提高转换速度，同时避免解析错误
date_format = '%m/%d/%Y'
df['Order Date'] = pd.to_datetime(df['Order Date'], format=date_format, errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format=date_format, errors='coerce')

# 检查转换失败的日期
failed_order_dates = df['Order Date'].isna().sum()
failed_ship_dates = df['Ship Date'].isna().sum()
print(f"订单日期转换失败数：{failed_order_dates}")
print(f"发货日期转换失败数：{failed_ship_dates}")
# ====================== 2. 邮编类型转换（重点） ======================
print("\n【2/3】邮编类型转换")
# 邮编虽然是数字，但应该作为字符串处理，否则会丢失前导零
# 先转换为字符串，再补全5位（美国邮编是5位数字）
df['Postal Code'] = df['Postal Code'].astype(str).str.split('.').str[0].str.zfill(5)
print("✅ 邮编已转换为字符串类型，并补全5位前导零")

# ====================== 3. 分类型列转换（节省内存+提高分析速度） ======================
print("\n【3/3】分类型列转换")
categorical_columns = [
    'Ship Mode', 'Segment', 'Country', 'City', 'State',
    'Region', 'Category', 'Sub-Category'
]

for col in categorical_columns:
    df[col] = df[col].astype('category')
    print(f"✅ 列 '{col}' 已转换为category类型")

# 提取年份列，方便后续按年分析
df['Year'] = df['Order Date'].dt.year

In [ ]:
print("="*60)
print("✅ 数据类型转换验证报告")
print("="*60)

# ====================== 1. 整体数据类型对比 ======================
print("\n【1/4】转换前后数据类型对比")
dtype_comparison = pd.DataFrame({
    '列名': df.columns,
    '转换前类型': [original_dtypes.get(col, '新增列') for col in df.columns],
    '转换后类型': df.dtypes.astype(str)
})
# 只显示类型发生变化的列
dtype_comparison = dtype_comparison[dtype_comparison['转换前类型'] != dtype_comparison['转换后类型']]
print(dtype_comparison.to_string(index=False))

# ====================== 2. 日期转换验证 ======================
print("\n【2/4】日期转换验证")
print(f"最早订单日期：{df['Order Date'].min().strftime('%Y-%m-%d')}")
print(f"最晚订单日期：{df['Order Date'].max().strftime('%Y-%m-%d')}")
print(f"最早发货日期：{df['Ship Date'].min().strftime('%Y-%m-%d')}")
print(f"最晚发货日期：{df['Ship Date'].max().strftime('%Y-%m-%d')}")

# 检查发货日期是否早于订单日期（逻辑错误）
invalid_dates = df[df['Ship Date'] < df['Order Date']]
if len(invalid_dates) == 0:
    print("✅ 所有发货日期都晚于或等于订单日期，逻辑正确")
else:
    print(f"❌ 发现 {len(invalid_dates)} 条发货日期早于订单日期的错误数据")

# ====================== 3. 邮编转换验证 ======================
print("\n【3/4】邮编格式验证")
# 检查所有邮编是否都是5位数字
invalid_postal_codes = df[~df['Postal Code'].str.match(r'^\d{5}$')]
if len(invalid_postal_codes) == 0:
    print("✅ 所有邮编都是5位数字，格式正确")
else:
    print(f"❌ 发现 {len(invalid_postal_codes)} 个格式错误的邮编")
    print(invalid_postal_codes['Postal Code'].unique())

# ====================== 4. 分类型列验证 ======================
print("\n【4/4】分类型列唯一值验证")
print("商品大类：", df['Category'].cat.categories.tolist())
print("发货方式：", df['Ship Mode'].cat.categories.tolist())
print("客户细分：", df['Segment'].cat.categories.tolist())
print("地区：", df['Region'].cat.categories.tolist())

print("\n" + "="*60)
print("🎉 所有验证通过！数据已准备好进行后续分析")
print("="*60)

In [ ]:
#时间
print("\n【问题5】数据时间范围")
earliest_order = df['Order Date'].min()
latest_order = df['Order Date'].max()
time_span = latest_order - earliest_order
print(f"最早订单日期：{earliest_order.strftime('%Y-%m-%d')}")
print(f"最晚订单日期：{latest_order.strftime('%Y-%m-%d')}")
print(f"数据覆盖时长：{time_span.days} 天（约 {round(time_span.days/365, 1)} 年）")

In [ ]:
#商品类别及其子类
print("\n" + "="*60)
print("\n【问题6】商品类别和子类别")
print("📦 商品大类(Category)：")
categories = df['Category'].unique()
for i, cat in enumerate(categories, 1):
    print(f"  {i}. {cat}")
print("\n📦 商品子类(Sub-Category)：")
subcategories = df.groupby('Category')['Sub-Category'].unique()
for cat, subs in subcategories.items():
    print(f"  {cat}：{', '.join(subs)}")
print("\n📊 各子类订单数量：")
sub_count = df['Sub-Category'].value_counts().sort_values(ascending=False)
print(sub_count.to_string())
print("\n" + "="*60)
print("✅ 探索性分析完成")
print("="*60)

#数据质量结论
没有缺失值，没有重复值，
销售额极值：存在 127 条销售额离群订单（占比 1.27%），均为单笔金额超过 2099.59 元的大额交易；同时存在单笔销售额低至 0.44 元的异常记录，需确认是否为赠品、样品或录入错误。
利润亏损问题：共 1871 条订单出现亏损（占比 18.7%），最低单笔利润达 - 6599.98 元
利润离群值：存在 107 条利润离群订单（占比 1.07%），包含大额盈利和大额亏损两类极端情况。
#折扣异常：存在 300 条折扣离群订单（占比 3.0%），均为折扣率超过 78% 的超高折扣交易。
总体评估：数据质量可靠，所有异常均有明确的业务解释，无影响分析结论的系统性错误。数据可直接进入深度分析阶段。

In [ ]:
# 1. 四年总销售额
total_sales = df['Sales'].sum()
print(f"1. 四年总销售额：${total_sales:,.2f}")

# 2. 品类销售额汇总
category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print(f"\n2. 各品类销售额（从高到低）：")
for cat, sales in category_sales.items():
    print(f"   {cat}：${sales:,.2f}")
print(f"→ 销售额最高品类：{category_sales.index[0]}")

# 3. 品类利润（是否亏损）
category_profit = df.groupby('Category')['Profit'].sum()
loss_cats = category_profit[category_profit < 0]
print(f"\n3. 各品类总利润：")
for cat, profit in category_profit.items():
    print(f"   {cat}：${profit:,.2f}")
if len(loss_cats) == 0:
    print("→ 所有品类均盈利，无亏损品类")
else:
    print(f"→ 亏损品类：{loss_cats.index.tolist()}")

# 4. 整体利润率
total_profit = df['Profit'].sum()
profit_rate = total_profit / total_sales * 100
print(f"\n4. 整体利润率：{profit_rate:.2f}%")
print("="*50)

In [ ]:
# ============================================================
# Phase 2: 整体画像 — 图1：年销售额与利润趋势
# ============================================================
yearly = df.groupby('Year').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum')
)

# 计算真实增长率
sales_growth = (yearly['销售额'].iloc[-1] / yearly['销售额'].iloc[0] - 1) * 100
profit_growth = (yearly['利润'].iloc[-1] / yearly['利润'].iloc[0] - 1) * 100

# 画图
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(yearly.index, yearly['销售额'], marker='o', color='#2E86AB', linewidth=2, label='销售额')
ax2.plot(yearly.index, yearly['利润'], marker='s', color='#A23B72', linewidth=2, label='利润')

ax1.set_xlabel('年份')
ax1.set_ylabel('销售额 (USD)', color='#2E86AB')
ax2.set_ylabel('利润 (USD)', color='#A23B72')
ax1.tick_params(axis='y', labelcolor='#2E86AB')
ax2.tick_params(axis='y', labelcolor='#A23B72')

ax1.set_title(f'2017年利润率回落至12.7%——近两年赚钱效率持续下降',
             fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('D:/Ecommerce-Data-Analysis/charts/phase2_overview.png', dpi=300, bbox_inches='tight')
plt.show()

# 打印增长率供确认
print(f'销售额增长率: {sales_growth:.1f}%')
print(f'利润增长率: {profit_growth:.1f}%')

In [ ]:
# 逐年利润率和增速对比（验证判断）
yearly['利润率'] = (yearly['利润'] / yearly['销售额'] * 100).round(2)
yearly['销售额同比增速'] = yearly['销售额'].pct_change().mul(100).round(1)
yearly['利润同比增速'] = yearly['利润'].pct_change().mul(100).round(1)

print('逐年核心指标：')
print(yearly[['利润率', '销售额同比增速', '利润同比增速']].to_string())

In [ ]:
# 按大类汇总销售额和利润
cat_stats = df.groupby('Category').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum')
)
cat_stats['利润率'] = (cat_stats['利润'] / cat_stats['销售额'] * 100).round(1)

# 画左右对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ---- 左图：销售额占比饼图 ----
colors_pie = ['#5B9BD5', '#ED7D31', '#A5A5A5']
explode = (0.02, 0.02, 0.02)
ax1.pie(cat_stats['销售额'], labels=cat_stats.index, autopct='%1.1f%%',
        colors=colors_pie, explode=explode, startangle=90)
ax1.set_title('Technology 和 Furniture 贡献近七成收入', fontsize=13, fontweight='bold')

# ---- 右图：利润贡献柱状图 ----
colors_bar = ['#27AE60' if v > 0 else '#E74C3C' for v in cat_stats['利润']]
bars = ax2.bar(cat_stats.index, cat_stats['利润'], color=colors_bar, width=0.5)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_title('Furniture 销售额排第二，利润却几乎为零', fontsize=13, fontweight='bold')
ax2.set_ylabel('利润 (USD)')

# 柱子上标注数值
for bar, profit, pct in zip(bars, cat_stats['利润'], cat_stats['利润率']):
    y_pos = bar.get_height() + 3000 if bar.get_height() > 0 else bar.get_height() - 8000
    ax2.text(bar.get_x() + bar.get_width()/2, y_pos,
             f'${profit:,.0f}\n({pct}%)', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase2_sales_profit_by_category.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 按年份+大类汇总利润率
year_cat = df.groupby(['Year', 'Category']).agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum')
)
year_cat['利润率'] = (year_cat['利润'] / year_cat['销售额'] * 100).round(1)
year_cat = year_cat.reset_index()

# 画分组柱状图
categories = ['Furniture', 'Office Supplies', 'Technology']
years = sorted(df['Year'].unique())
colors = ['#A23B72', '#2E86AB', '#F18F01']
bar_width = 0.25
x = np.arange(len(years))

fig, ax = plt.subplots(figsize=(10, 6))

for i, (cat, color) in enumerate(zip(categories, colors)):
    cat_data = year_cat[year_cat['Category'] == cat]
    bars = ax.bar(x + i * bar_width, cat_data['利润率'].values,
                  width=bar_width, color=color, label=cat)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', fontsize=8)

ax.set_xlabel('年份')
ax.set_ylabel('利润率 (%)')
ax.set_title('Furniture 利润率三年持续垫底，且 2017 年再度下滑', fontsize=13, fontweight='bold')
ax.set_xticks(x + bar_width)
ax.set_xticklabels(years)
ax.legend(loc='upper right')
ax.set_ylim(0, 20)

plt.tight_layout()
plt.savefig('../charts/phase2_profit_margin_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

## Phase 2 总结：整体画像
（自己写AI更改的版本）
2014-2017年，超市累计营收 $229.7 万、净利润 $28.6 万，整体利润率 12.5%。
但盈利能力正在恶化：2017年利润率从 13.4% 降至 12.7%，利润增速（14.2%）已落后于销售额增速（20.4%）。

核心问题出在 Furniture 品类：该品类贡献了 32.3% 的销售额（$74.2万），
但利润仅 $1.8 万，利润率为 2.5%，不到 Technology（17.4%）的六分之一。
Office Supplies 和 Technology 以相近的销售额分别创造了 $12.2 万和 $14.5 万利润，
Furniture 用同样的销售规模只赚了它们的零头。

结论：公司不是不赚钱，而是 Furniture 在拖累整体利润率。
下一步将深入拆解 Furniture 品类，从折扣、子类别、客户类型和地区四个维度定位根因。


## Phase 2 总结：整体画像
(AI给出标准答案)
四年卖了 $230 万，赚了 $29 万，整体利润率 12.5%。
数字不难看，但趋势在变坏——2017年利润增速（14.2%）跑输了收入增速（20.4%），说明公司卖得越多，赚得越少。

只有一个原因：Furniture（家具）。
相同规模的收入（$74万 vs $72万 vs $84万），Office Supplies 和 Technology 各赚了 $12-15 万，Furniture 只赚了 $1.8 万。利润率 2.5%，相当于每卖 $100 家具，到手只有 $2.5——还不够覆盖房租。

结论很清楚：公司健康，但背上长了一个肿瘤。不切掉或者不治疗，随着销售额继续增长，这个肿瘤会吃掉更多利润。

下一步：拆解 Furniture 品类，找根因。不做宽泛分析，直奔折扣、子品类和客户群。

In [ ]:
# ========== 折扣分析：家具是不是打折打太狠？ ==========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 左图：三个品类的平均折扣率对比
avg_discount = df.groupby('Category')['Discount'].mean().sort_values() * 100
colors_d = ['#E74C3C' if cat == 'Furniture' else '#95A5A6' for cat in avg_discount.index]
bars = ax1.bar(avg_discount.index, avg_discount.values, color=colors_d, width=0.5)
ax1.set_title('Furniture 平均折扣 18%，远超另外两个品类', fontsize=13, fontweight='bold')
ax1.set_ylabel('平均折扣率 (%)')
for bar, val in zip(bars, avg_discount.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')

# 右图：Furniture 每笔订单的折扣 vs 利润
furniture = df[df['Category'] == 'Furniture']
ax2.scatter(furniture['Discount'] * 100, furniture['Profit'], alpha=0.5, c='#E74C3C', s=15)
ax2.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('折扣率 (%)')
ax2.set_ylabel('利润 (USD)')
ax2.set_title('Furniture：折扣 > 30% 的订单几乎全在亏钱', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase3_discount.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ========== 地区分析：Furniture 在哪些地区赚钱？哪些在亏？ ==========
region_stats = furniture.groupby('Region').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum')
)
region_stats['利润率'] = (region_stats['利润'] / region_stats['销售额'] * 100).round(1)
region_stats = region_stats.sort_values('利润', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 左图：Furniture 销售额的地区分布
ax1.pie(region_stats['销售额'], labels=region_stats.index, autopct='%1.1f%%',
        colors=['#5B9BD5','#ED7D31','#A5A5A5','#FFC000'], startangle=90)
ax1.set_title('Furniture 销售额地区分布：东西部占七成', fontsize=13, fontweight='bold')

# 右图：各地区的利润和利润率
colors_r = ['#27AE60' if v > 0 else '#E74C3C' for v in region_stats['利润']]
bars = ax2.bar(region_stats.index, region_stats['利润'], color=colors_r, width=0.5)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_title('Central 地区是唯一亏损的大区', fontsize=13, fontweight='bold')
ax2.set_ylabel('利润 (USD)')
for bar, profit, pct in zip(bars, region_stats['利润'], region_stats['利润率']):
    y_pos = bar.get_height() + 500 if bar.get_height() > 0 else bar.get_height() - 1500
    ax2.text(bar.get_x() + bar.get_width()/2, y_pos,
             f'${profit:,.0f}\n({pct}%)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase3_region.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ========== 客户类型分析：谁在买家具？谁拿到了最高折扣？ ==========
seg_stats = furniture.groupby('Segment').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum'),
    平均折扣=('Discount', 'mean')
).sort_values('销售额', ascending=False)
seg_stats['利润率'] = (seg_stats['利润'] / seg_stats['销售额'] * 100).round(1)
seg_stats['平均折扣率'] = (seg_stats['平均折扣'] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 左图：销售额占比
ax1.pie(seg_stats['销售额'], labels=seg_stats.index, autopct='%1.1f%%',
        colors=['#5B9BD5','#ED7D31','#A5A5A5'], startangle=90)
ax1.set_title('消费者（Consumer）是家具最大买家', fontsize=13, fontweight='bold')

# 右图：各客户类型的利润率 + 折扣率
x = np.arange(len(seg_stats))
width = 0.35
bars1 = ax2.bar(x - width/2, seg_stats['利润率'], width, color='#2E86AB', label='利润率(%)')
bars2 = ax2.bar(x + width/2, seg_stats['平均折扣率'], width, color='#E74C3C', label='平均折扣率(%)')
ax2.set_xticks(x)
ax2.set_xticklabels(seg_stats.index)
ax2.set_title('Consumer 利润率最低，折扣率却最高', fontsize=13, fontweight='bold')
ax2.legend()
for bar in bars1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase3_segment.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ========== Central 地区深挖：客户结构 + 折扣 ==========
central = furniture[furniture['Region'] == 'Central']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 左图：Central 各客户类型的利润贡献
seg_central = central.groupby('Segment').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum'),
    平均折扣=('Discount', 'mean')
)
seg_central['利润率'] = (seg_central['利润'] / seg_central['销售额'] * 100).round(1)
seg_central['平均折扣率'] = (seg_central['平均折扣'] * 100).round(1)

colors_sc = ['#27AE60' if v > 0 else '#E74C3C' for v in seg_central['利润']]
bars = ax1.bar(seg_central.index, seg_central['利润'], color=colors_sc, width=0.5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_title('Central 地区：Consumer 是亏损主力', fontsize=13, fontweight='bold')
ax1.set_ylabel('利润 (USD)')
for bar, p, pct, d in zip(bars, seg_central['利润'], seg_central['利润率'], seg_central['平均折扣率']):
    y_pos = bar.get_height() + 200 if bar.get_height() > 0 else bar.get_height() - 800
    ax1.text(bar.get_x() + bar.get_width()/2, y_pos,
             f'${p:,.0f}\n利润率{pct}%\n折扣{d}%', ha='center', fontsize=8, fontweight='bold')

# 右图：Central vs 其他地区 Consumer 折扣对比
all_consumer = furniture[furniture['Segment'] == 'Consumer']
all_consumer_regions = all_consumer.groupby('Region')['Discount'].mean().sort_values() * 100
colors_cr = ['#E74C3C' if reg == 'Central' else '#95A5A6' for reg in all_consumer_regions.index]
bars2 = ax2.bar(all_consumer_regions.index, all_consumer_regions.values, color=colors_cr, width=0.5)
ax2.set_title('Consumer 客户折扣率：Central 排第一', fontsize=13, fontweight='bold')
ax2.set_ylabel('Consumer 平均折扣率 (%)')
for bar, val in zip(bars2, all_consumer_regions.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase3_central_deep.png', dpi=300, bbox_inches='tight')
plt.show()

# 打印关键数字
print('Central 地区家具利润:', central['Profit'].sum().round(2))
print('Central Consumer 利润:', central[central['Segment']=='Consumer']['Profit'].sum().round(2))
print('Central Consumer 折扣率:', (central[central['Segment']=='Consumer']['Discount'].mean()*100).round(1), '%')


In [ ]:
# ========== Central + Consumer：哪个子品类在亏钱？ ==========
target = furniture[(furniture['Region'] == 'Central') & (furniture['Segment'] == 'Consumer')]

sub_central = target.groupby('Sub-Category').agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum'),
    订单数=('Order ID', 'nunique'),
    平均折扣=('Discount', 'mean')
)
sub_central['利润率'] = (sub_central['利润'] / sub_central['销售额'] * 100).round(1)
sub_central['平均折扣率'] = (sub_central['平均折扣'] * 100).round(1)
sub_central = sub_central.sort_values('利润')

fig, ax = plt.subplots(figsize=(10, 5))
colors_sub = ['#E74C3C' if v < 0 else '#27AE60' for v in sub_central['利润']]
bars = ax.barh(sub_central.index, sub_central['利润'], color=colors_sub)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Central Consumer：Tables 和 Bookcases 是亏损黑洞', fontsize=13, fontweight='bold')
ax.set_xlabel('利润 (USD)')
for bar, profit, pct, disc in zip(bars, sub_central['利润'], sub_central['利润率'], sub_central['平均折扣率']):
    x_pos = bar.get_width() + 20 if bar.get_width() > 0 else bar.get_width() - 150
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'${profit:,.0f} | 利润率{pct}% | 折扣{disc}%', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('../charts/phase3_central_consumer_subcat.png', dpi=300, bbox_inches='tight')
plt.show()

print('各子品类明细：')
print(sub_central.to_string())


## Phase 3 结论

Furniture 利润率仅 2.5%，根因是一条清晰的连锁反应：

1. 折扣失控：Furniture 平均折扣 17.4%，是另外两个品类的约 2 倍。折扣 > 30% 的订单几乎全是亏损。
2. 地区塌方：Central 地区是唯一亏损大区，四年亏了 $2,871。该地区 Consumer 客户平均折扣高达 31.4%。
3. 三个黑洞子品类：Tables（亏$3,964，折扣 31.2%）、Furnishings（亏$1,454，折扣 41.1%）、
   Bookcases（亏$1,497，折扣 26.1%）是 Central Consumer 的主要亏损来源。

关键证据：同样的 Central 地区、同样的 Consumer 客户，Chairs 赚了 $2,921（利润率 6.8%）。
这说明问题不在"个人买家买不起"，而在桌子、书架、装饰品被打折打到失血。

根因推测：
1.员工有较大的自主定价权，薪酬与销售额而非利润挂钩，
导致其倾向于用高折扣换取高销售额，尤其是高单价家具品类。
这解释了为什么 Finance 类产品折扣率 41%——越是单价高的产品，员工越需要用折扣去"说服"客户成交。可能性 
2：定价策略问题（不是折扣打太多，是标价本来就虚高）
Chairs 可以打 20% 折扣还赚钱，Tables 打 31% 就亏——会不会 Chairs 的标价本身就有更高的利润空间？ 如果 Tables 进价 $500 标价 $700，打 7 折就亏；但如果 Chairs 进价 $300 标价 $900，打 5 折还赚。
可能性 3：物流成本差异
一张桌子 50 公斤，一把椅子 15 公斤。同一辆卡车送同样的距离，桌子的运费可能吃掉全部利润。 你的数据里有 Ship Mode 字段——如果 Tables 大量使用 Standard Class（卡车运输），而 Chairs 能用更便宜的配送方式，那亏钱不是折扣的问题，是物流的问题。
可能性 4：地区消费能力差异
Central 地区（大概是芝加哥、达拉斯这些中西部城市）的消费者真的买得起没有折扣的家具吗？ 有可能这个地区的客户对价格极度敏感，不给 30% 折扣根本不买。如果是这样，"限制折扣上限到 25%"的建议会直接导致销量归零。
下一步建议：
1. 对 Central 地区 Furniture 品类设置折扣上限（建议不超过 25%）
2. 将销售激励从纯销售额考核改为"销售额 + 利润率"双维度考核
3. 对比 Chairs 和 Tables 的定价策略，判断 Tables 是否本身就定价虚高
检验方法：对比每种子品类的原始标价（Sales ÷ Quantity）和最终利润率的关系。
检验方法：对比 Chairs 和 Tables 的 Ship Mode 分布和该配送方式下的利润率
检验方法：对比 Central 地区折扣 > 30% 的订单和折扣 < 20% 的订单在销售量上的差异——看看不打折是不是真的卖不动。

定价模拟验证：

以 Chairs（利润率 6.8%，成本加成率 1.3x）为健康基准，对另外三个子品类进行定价模拟：
- Bookcases：若采用相同加成率，利润率可从 -3.0% 扭转为 +3.3%。问题在定价过低。
- Furnishings：当前利润率 14.2%，即便折扣 41% 仍盈利。折扣虽高但利润空间充足，无需干预。
- Tables：当前售价已低于成本（单价 $167 < 成本 $181），即便套用 Chairs 加成率仍亏。
  只有加成率提升 + 折扣降至 17% 双管齐下，才能扭亏。

Tables 是 Furniture 最核心的问题：成本高、定价低、折扣大，三重叠加。
建议优先审计 Tables 的采购成本和供应商合同。


In [ ]:
# ========== 验证可能性2：定价策略对比 ==========
# 筛选 Furniture 四个子品类
subs = ['Chairs', 'Tables', 'Bookcases', 'Furnishings']
sub_data = furniture[furniture['Sub-Category'].isin(subs)]

# 计算每个子品类的核心指标
sub_pricing = sub_data.groupby('Sub-Category').agg(
    总销售额=('Sales', 'sum'),
    总利润=('Profit', 'sum'),
    总销量=('Quantity', 'sum'),
    平均折扣=('Discount', 'mean')
)
sub_pricing['平均单价'] = (sub_pricing['总销售额'] / sub_pricing['总销量']).round(2)
sub_pricing['单件利润'] = (sub_pricing['总利润'] / sub_pricing['总销量']).round(2)
sub_pricing['单件成本'] = (sub_pricing['平均单价'] - sub_pricing['单件利润']).round(2)
sub_pricing['利润率'] = (sub_pricing['总利润'] / sub_pricing['总销售额'] * 100).round(1)
sub_pricing['折扣率'] = (sub_pricing['平均折扣'] * 100).round(1)

print('=== 四个子品类的定价与利润结构 ===')
print(sub_pricing.to_string())

# ========== 模拟：如果 Chairs 被打到 Tables 的折扣水平 ==========
chair_row = sub_pricing.loc['Chairs']
table_row = sub_pricing.loc['Tables']

# Chairs 当前：折扣20%，单价 = 原价 × (1-20%)
# 推算 Chairs 原价
chair_orig_price = chair_row['平均单价'] / (1 - chair_row['平均折扣'])
# 如果 Chairs 打到 Tables 的折扣（31.2%），新单价
chair_new_price = chair_orig_price * (1 - table_row['平均折扣'])
# 新单件利润
chair_new_profit_per_unit = chair_new_price - chair_row['单件成本']
chair_new_margin = (chair_new_profit_per_unit / chair_new_price * 100)

print(f'\n=== 模拟：如果 Chairs 被打到 Tables 的折扣水平 ===')
print(f'Chairs 推算原价: ${chair_orig_price:.2f}')
print(f'Chairs 当前折扣: {chair_row["折扣率"]:.1f}% → 当前单价: ${chair_row["平均单价"]:.2f} → 当前单件利润: ${chair_row["单件利润"]:.2f}')
print(f'Chairs 模拟折扣: {table_row["折扣率"]:.1f}% → 模拟单价: ${chair_new_price:.2f} → 模拟单件利润: ${chair_new_profit_per_unit:.2f}')
print(f'当前利润率: {chair_row["利润率"]:.1f}% → 模拟利润率: {chair_new_margin:.1f}%')
print(f'结论: Chairs {"仍然盈利" if chair_new_profit_per_unit > 0 else "会亏损"}')


In [ ]:
# ========== 验证：如果三个亏损子品类按 Chairs 的定价标准定价，还会亏吗？ ==========
subs = ['Chairs', 'Tables', 'Bookcases', 'Furnishings']
sub_data = furniture[furniture['Sub-Category'].isin(subs)]

# 每个子品类的单价、利润、成本、折扣
sub_stats = sub_data.groupby('Sub-Category').agg(
    总销售额=('Sales', 'sum'),
    总利润=('Profit', 'sum'),
    总销量=('Quantity', 'sum'),
    平均折扣=('Discount', 'mean')
)
sub_stats['平均单价'] = sub_stats['总销售额'] / sub_stats['总销量']
sub_stats['单件利润'] = sub_stats['总利润'] / sub_stats['总销量']
sub_stats['单件成本'] = sub_stats['平均单价'] - sub_stats['单件利润']

# Chairs 的定价标准：成本加成率 = 原价 / 成本
chair_row = sub_stats.loc['Chairs']
chair_orig_price = chair_row['平均单价'] / (1 - chair_row['平均折扣'])  # 推算原价
chair_markup = chair_orig_price / chair_row['单件成本']                 # 原价是成本的几倍

print(f'=== Chairs 定价标准 ===')
print(f'平均单价: ${chair_row["平均单价"]:.2f}')
print(f'单件成本: ${chair_row["单件成本"]:.2f}')
print(f'单件利润: ${chair_row["单件利润"]:.2f}')
print(f'折扣率: {chair_row["平均折扣"]*100:.1f}%')
print(f'成本加成率: {chair_markup:.1f}x（原价 = 成本 × {chair_markup:.1f}）')

# ========== 模拟：给其他三个品类套上 Chairs 的加成率 ==========
print(f'\n=== 模拟结果：如果都用 Chairs 的 {chair_markup:.1f}x 加成率 ===')
print(f'{"品类":<15s} {"当前单价":>8s} {"当前利润":>8s} {"当前利润率":>8s}  {"模拟单价":>8s} {"模拟利润":>8s} {"模拟利润率":>8s}  {"结论"}')

for sub in ['Tables', 'Bookcases', 'Furnishings']:
    row = sub_stats.loc[sub]
    # 用 Chairs 加成率重新定价
    new_orig_price = row['单件成本'] * chair_markup
    # 保持各自当前的折扣水平
    new_actual_price = new_orig_price * (1 - row['平均折扣'])
    new_profit = new_actual_price - row['单件成本']
    new_margin = new_profit / new_actual_price * 100

    old_price = row['平均单价']
    old_profit = row['单件利润']
    old_margin = row['总利润'] / row['总销售额'] * 100

    result = '✓ 扭亏' if new_profit > 0 else '✗ 仍亏损'
    print(f'{sub:<15s} ${old_price:>7.2f} ${old_profit:>7.2f} {old_margin:>7.1f}%  ${new_actual_price:>7.2f} ${new_profit:>7.2f} {new_margin:>7.1f}%  {result}')

# ========== 更激进的模拟：如果折扣也降到 Chairs 水平 ==========
print(f'\n=== 激进模拟：Chairs 加成率 + Chairs 折扣水平 ===')
chair_discount = chair_row['平均折扣']
print(f'{"品类":<15s} {"模拟单价":>8s} {"模拟利润":>8s} {"模拟利润率":>8s}  {"结论"}')

for sub in ['Tables', 'Bookcases', 'Furnishings']:
    row = sub_stats.loc[sub]
    new_orig_price = row['单件成本'] * chair_markup
    new_actual_price = new_orig_price * (1 - chair_discount)
    new_profit = new_actual_price - row['单件成本']
    new_margin = new_profit / new_actual_price * 100

    result = '✓ 扭亏' if new_profit > 0 else '✗ 仍亏损'
    print(f'{sub:<15s} ${new_actual_price:>7.2f} ${new_profit:>7.2f} {new_margin:>7.1f}%  {result}')


In [ ]:
# ========== 验证可能性3：Chairs vs Tables 的配送方式差异 ==========
target_subs = furniture[furniture['Sub-Category'].isin(['Chairs', 'Tables'])]

# 按子品类 + 配送方式汇总
ship_compare = target_subs.groupby(['Sub-Category', 'Ship Mode']).agg(
    订单数=('Order ID', 'nunique'),
    总销售额=('Sales', 'sum'),
    总利润=('Profit', 'sum'),
    平均折扣=('Discount', 'mean')
)
ship_compare['利润率'] = (ship_compare['总利润'] / ship_compare['总销售额'] * 100).round(1)

# 计算每种配送方式的占比
ship_compare['配送占比'] = (ship_compare['订单数'] / ship_compare.groupby('Sub-Category')['订单数'].transform('sum') * 100).round(1)

print('=== Chairs vs Tables：各配送方式的利润表现 ===')
print(ship_compare.to_string())

# 核心对比：同一种配送方式下，Chairs 和 Tables 的利润率差多少
print('\n=== 同配送方式下的利润率对比 ===')
for mode in ['Standard Class', 'Second Class', 'First Class', 'Same Day']:
    chairs_mode = ship_compare.loc[('Chairs', mode)] if ('Chairs', mode) in ship_compare.index else None
    tables_mode = ship_compare.loc[('Tables', mode)] if ('Tables', mode) in ship_compare.index else None
    if chairs_mode is not None and tables_mode is not None:
        chairs_pct = chairs_mode['利润率']
        tables_pct = tables_mode['利润率']
        chairs_share = chairs_mode['配送占比']
        tables_share = tables_mode['配送占比']
        print(f'{mode}: Chairs利润率 {chairs_pct}% (占比{chairs_share}%) | Tables利润率 {tables_pct}% (占比{tables_share}%) | 利润率差距 {chairs_pct - tables_pct:.1f}%')


In [ ]:
# ========== 验证可能性4：Central 地区不打折就卖不动？ ==========
central_f = furniture[furniture['Region'] == 'Central']

# 分三组：低折扣（<20%）、中折扣（20-30%）、高折扣（>30%）
central_f['折扣分组'] = pd.cut(central_f['Discount'] * 100,
                              bins=[0, 20, 30, 100],
                              labels=['折扣<20%', '折扣20-30%', '折扣>30%'])

discount_groups = central_f.groupby('折扣分组').agg(
    订单数=('Order ID', 'nunique'),
    总销售额=('Sales', 'sum'),
    总利润=('Profit', 'sum'),
    总销量=('Quantity', 'sum')
)
discount_groups['订单占比'] = (discount_groups['订单数'] / discount_groups['订单数'].sum() * 100).round(1)
discount_groups['利润率'] = (discount_groups['总利润'] / discount_groups['总销售额'] * 100).round(1)

print('=== Central 地区：不同折扣水平的销售量 ===')
print(discount_groups.to_string())

print('\n关键问题：Central 地区多少订单是靠高折扣换来的？')
high_disc_pct = discount_groups.loc['折扣>30%', '订单占比']
print(f'折扣>30%的订单占比: {high_disc_pct}%')
if high_disc_pct > 50:
    print('→ 超过一半订单依赖高折扣。限制折扣上限可能导致销量大幅下滑。')
else:
    print(f'→ 仅 {high_disc_pct}% 的订单需要 >30% 折扣，限制折扣不会影响大部分销售。')

# 对比：Central vs 其他地区的折扣分布
other_regions = furniture[furniture['Region'] != 'Central']
central_avg_disc = central_f['Discount'].mean() * 100
other_avg_disc = other_regions['Discount'].mean() * 100
central_orders = central_f['Order ID'].nunique()
other_orders = other_regions['Order ID'].nunique()
print(f'\nCentral 平均折扣: {central_avg_disc:.1f}%（{central_orders}单）')
print(f'其他地区平均折扣: {other_avg_disc:.1f}%（{other_orders}单）')


In [ ]:
# 基础参数
unit_cost = 181.05
current_unit_price = 166.77
current_unit_profit = current_unit_price - unit_cost
sim_unit_price = 197.01
sim_unit_profit = sim_unit_price - unit_cost
base_sales = 100  # 基准销量

# 不同下滑比例
decline_rates = [0.3, 0.5, 0.7, 0.8, 0.9]
scenarios = []

for rate in decline_rates:
    sim_sales = base_sales * (1 - rate)
    current_total_profit = base_sales * current_unit_profit
    sim_total_profit = sim_sales * sim_unit_profit
    profit_diff = sim_total_profit - current_total_profit
    scenarios.append({
        "销量下滑比例": f"{int(rate*100)}%",
        "模拟销量": int(sim_sales),
        "当前总利润": round(current_total_profit, 2),
        "模拟总利润": round(sim_total_profit, 2),
        "利润改善额": round(profit_diff, 2)
    })

# 转换为DataFrame并打印
df = pd.DataFrame(scenarios)
print("不同销量下滑场景的利润对比（基准销量=100件）")
print(df)

# 计算盈亏平衡销量（考虑固定成本示例）
fixed_cost = 500  # 假设固定成本为500美元
break_even_sales = fixed_cost / sim_unit_profit
print(f"\n考虑固定成本${fixed_cost}时，盈亏平衡销量：{int(break_even_sales)+1}件")
print(f"对应最大允许销量下滑比例：{int((1 - break_even_sales/base_sales)*100)}%")

## Phase 4：商业建议

### 一、问题一句话
[用一句话说清楚：Furniture 品类四年只赚了多少钱，利润率多少，根因是什么]

### 二、两条可选路径

#### 路径A：止血方案 —— Central 地区下架 Tables
- **做什么**：[具体到地区、子品类、库存处理、客户过渡]
- **预期效果**：年度止损约 $[...]，但损失约 $[...] 销售额。其他 Furniture 子品类可承接约 [...]% 的转移购买。
- **风险1**：[至少一个]
- **风险2**：[至少一个]
- **适用条件**：什么情况下选这条？

#### 路径B：改造方案 —— 保留 Tables，提价+控折扣
- **做什么**：分 [...] 步执行。第一步：[...]；第二步：[...]；第三步：[...]
- **目标定价**：成本加成率从 [...] 提升至 1.3x，折扣上限从 [...]% 降至 [...]%，单件利润从 -$14.28 变为 +$[...]
- **预期效果**：即使销量下降 [...]%，年利润仍优于当前
- **风险1**：[至少一个]
- **风险2**：[至少一个]
- **适用条件**：什么情况下选这条？

### 三、配套措施（不管选哪条都要做）
1. **考核机制改革**：[...]
2. **供应商审计**：[...]
3. **Bookcases 同步调价**：[...]

### 四、执行风险与监控指标
- **先行指标**（执行后1-3个月看什么）：[...]
- **滞后指标**（执行后6-12个月看什么）：[...]
- **红线**：如果 [...] 就立即停止/回调

### 五、优先级排序
| 优先级 | 行动 | 责任人建议 | 时间窗口 |
|--------|------|-----------|----------|
| P0（立即） | [...] | [...] | [...] |
| P1（本季度） | [...] | [...] | [...] |
| P2（下季度） | [...] | [...] | [...] |


# Phase 4：商业建议

## 一、问题一句话
Furniture 品类四年累计盈利仅 **$18,451**，平均利润率 **2.5%**（远低于 Office Supplies 的 17.0% 和 Technology 的 17.4%）。根因是折扣失控：Tables 在全公司四个地区中三个在亏（总亏 $17,725），Con​sumer 客户利润率仅 1.8%。

## 二、两条可选路径

### 路径A：止血方案 —— Central 地区下架 Tables
- **做什么**：
  1. 立即停止 Central 地区 Tables 销售，不接新订单
  2. 将 Central 库存转移至 **West 地区**（唯一 Tables 盈利区，利润率 1.7%），East 和 South 同步启动定价改革
  3. 剩余库存以 ≤10% 折扣在 Central 限时清仓
  4. 对已下单未发货客户，引导更换为 Chairs 或提供 10% 额外折扣补偿
- **预期效果**：Central 年均止损约 **$890**，但损失约 **$9,789** 年销售额。Chairs 可承接部分转移购买
- **风险1**：价格敏感 Consumer 客户流失，Central 地区 Furniture 份额下降 10%-15%
- **风险2**：库存转移不畅导致额外 $1,000-$2,000 损失
- **风险3**：Tables 在 East 亏损 $11,025（全公司最严重，-28.2%），仅砍 Central 不能解决系统性问题
- **适用条件**：公司需要快速止血；或改造方案执行 6 个月后 Tables 销量下滑超 80%

### 路径B：改造方案 —— 保留 Tables，全公司统一提价+控折扣
- **做什么**：分 3 步渐进执行，适用于全公司四个地区
  1. **第 1 个月（测试期）**：原价加成率从 1.25x 微调至 1.31x（≈Chairs 基准），关键不是涨原价而是**控折扣**——折扣上限从 26.1% 降至 25%；同步推出"免费安装+1年延长质保"非价格促销
  2. **第 2-3 个月（调整期）**：若销量下滑 ≤50%，折扣上限进一步降至 20%
  3. **第 4-6 个月（稳定期）**：稳定在 Chairs 标准的 1.31x 加成率 + 17% 折扣
- **目标定价**：成本 $181.06，新单价 **$197.01**，单件利润从 -$14.28 变为 **+$15.96**
- **预期效果（Central 地区）**：当前年亏损 -$890，扭亏后年利润 +$314（70% 下滑场景）～+$1,045（无下滑）；加 $5/件促销成本后仍为正利润
- **风险1**：Central 55.4% 订单依赖 >30% 折扣，折扣从 26% 降到 17% 可能引发销量骤降超 80%
- **风险2**：East 地区（Tables 亏损最严重，-$11,025）折扣结构可能同样依赖高折扣，需单独评估
- **适用条件**：公司愿承受短期销量下滑换取长期盈利健康；非价格促销能部分抵消价格敏感。**本方案应作为全公司 Tables 统一改革方案，而非仅限 Central。**

## 三、配套措施（不管选哪条都要做）
1. **考核机制改革**：销售考核从"销售额 60%+利润 20%"调整为"**利润 50%+折扣率 20%+销售额 20%**"，Central 和 East 地区优先执行
2. **供应商审计**：Tables 单件成本 $181.06，启动供应商谈判目标降本 5%-10%
3. **Bookcases 同步调价**：同样提升至 1.31x 加成率 + 17% 折扣上限，解决其 $1,497 亏损

## 四、执行风险与监控指标
- **先行指标（1-3 个月）**：Tables 周度销量、客户投诉率、竞争对手调价、非价格促销转化率
- **滞后指标（6-12 个月）**：Tables 单品利润率、各地区 Furniture 总利润、Consumer 留存率
- **红线**：Tables 连续 3 个月销量下滑超 **80%**，或全公司 Furniture 总利润低于当前 $18,451，立即回调

## 五、优先级排序
| 优先级 | 行动 | 责任人 | 时间 |
|--------|------|--------|------|
| P0（立即） | 全公司 Tables 折扣上限降至 25%；发布新销售考核机制 | 销售总监+品类经理 | 1 周内 |
| P1（本季度） | Tables 供应商审计与成本谈判；Bookcases 同步调价 | 采购经理 | 3 个月内 |
| P2（下季度） | Tables 加成率逐步提升至 1.31x；评估路径 A vs B 长期策略 | 品类经理+区域总监 | 6 个月内 |

In [ ]:
# ========== Phase 4 支撑验证 ==========
furniture = df[df['Category'] == 'Furniture']

# 1. Tables 在各地区的盈亏
print('=== Q1: Tables 在所有地区的利润率 ===')
tables_by_region = furniture[furniture['Sub-Category'] == 'Tables'].groupby('Region', observed=True).agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum'),
    订单数=('Order ID', 'nunique')
)
tables_by_region['利润率'] = (tables_by_region['利润'] / tables_by_region['销售额'] * 100).round(1)
print(tables_by_region.to_string())

# 2. Furniture 各 Segment 的利润率
print('\n=== Q2: Furniture 各客户类型利润率 ===')
seg_stats = furniture.groupby('Segment', observed=True).agg(
    销售额=('Sales', 'sum'),
    利润=('Profit', 'sum'),
    平均折扣=('Discount', 'mean')
)
seg_stats['利润率'] = (seg_stats['利润'] / seg_stats['销售额'] * 100).round(1)
seg_stats['折扣率'] = (seg_stats['平均折扣'] * 100).round(1)
print(seg_stats[['利润率', '折扣率']].to_string())

# 3. Tables 扭亏模拟 + 促销成本
print('\n=== Q3: Tables 扭亏方案敏感度 ===')
tables_all = furniture[furniture['Sub-Category'] == 'Tables']
cost_per_unit = (tables_all['Sales'].sum() / tables_all['Quantity'].sum()) - (tables_all['Profit'].sum() / tables_all['Quantity'].sum())
new_price = cost_per_unit * 1.311 * (1 - 0.17)
base_profit = new_price - cost_per_unit

print(f'Tables 单件成本: ${cost_per_unit:.2f}')
print(f'基准单件利润（无促销成本）: ${base_profit:.2f}')
for promo in [5, 7.5, 10]:
    net = base_profit - promo
    print(f'  促销成本 ${promo}/件 -> 净利润 ${net:.2f}')

# Central Tables 年化分析
ct_profit = tables_by_region.loc['Central', '利润']
ct_units = tables_all[tables_all['Region'] == 'Central']['Quantity'].sum()
annual_loss = ct_profit / 4
print(f'\nCentral Tables 年化亏损: ${annual_loss:,.2f}（{ct_units} 件 / 4年）')

for pct in [0, 30, 50, 70, 90]:
    rem = (ct_units / 4) * (1 - pct / 100)
    new_p = rem * base_profit
    print(f'销量下滑{pct}%: 年销量{rem:.0f}件 -> 年利润${new_p:,.2f}（当前${annual_loss:,.2f}）')